# 입찰메이트 서빙평가 — 타입 라우팅 (A/B/D/E = chunks_all, C = kh_v3)

KURE + Phi-4-mini(FT LoRA) E2E 평가. 타입에 따라 두 retriever 를 스왑하며 579행 생성.

- **A/B/D/E** → `chunks_all` (`bidmate_chunks_all`, 기관키 `agency`)
- **C** → `kh_v3` (`bidmate_kh_v3_KURE_PHI`, 기관키 auto, chunk_id ← child_id)

Drive 의 `retrieval.py` / `generation.py` 는 **수정하지 않음**. 인메모리 패치로 처리.

실행 순서: `[0] → [0a] → [0b](선택) → [1] → [2] → [3] → [4] → [5] → [6] 스모크 → [7] 생성 → [8]~[14]`

In [ ]:
# [0] 설치
!pip install -q chromadb sentence-transformers rank_bm25 kiwipiepy peft transformers accelerate openai tqdm nest_asyncio rapidfuzz
!pip uninstall -y torchao
print("설치 완료 — 런타임 재시작 메시지 뜨면 재시작 후 [0a]부터")

In [ ]:
# [0a] 드라이브 마운트 + bidmate 핵심 파일 존재/크기 확인 (업로드 누락 조기 감지)
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/data/bidmate'

NEED = [
    # chunks_all (A/B/D/E)
    'chroma_db.tar.gz',
    'chunks/chunks_all.json',
    'bm25/bm25_index_bidmate_chunks_all_A-2.pkl',
    # kh_v3 (C)
    'kh_v3_chroma_FIXED.tar.gz',
    'chunks/kh_v3.json',
    'bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl',
    # 공통
    'eval/eval_retrieval_579.csv',
    'code/retrieval.py',
    'code/generation.py',
    'peft_output/phi4-mini/lora_adapter',
]
for need in NEED:
    fp = f'{DRIVE}/{need}'
    if os.path.exists(fp):
        sz = os.path.getsize(fp)/1e6 if os.path.isfile(fp) else 0
        print(f'OK       {sz:8.0f}MB  {need}' if sz else f'OK       <dir/small>  {need}')
    else:
        print(f'MISSING            {need}')

for t in ['chroma_db.tar.gz', 'kh_v3_chroma_FIXED.tar.gz']:
    fp=f'{DRIVE}/{t}'
    if os.path.exists(fp):
        sz=os.path.getsize(fp)/1e6
        print(f'\n{t}: {sz:.0f}MB', '✅' if sz>1 else '⚠️ 너무 작음 — 업로드 미완 의심')

In [ ]:
# [0b] 진단(선택) — 두 tar 안의 컬렉션 이름·개수·기관키 확인 (마운트 후 1회)
#   count>0 인 이름을 [1] 의 COLL 에, 감지된 기관키를 AGENCY_KEY 에 반영.
import os, shutil, tarfile, gc, chromadb
DRIVE = '/content/drive/MyDrive/data/bidmate'
_AGENCY_CANDIDATES = ['agency','organization_cleaned','organization','agency_name','institution']

def _detect_agency_key(metas):
    for k in _AGENCY_CANDIDATES:
        if any(isinstance(m,dict) and m.get(k) for m in metas): return k
    for m in metas:
        if isinstance(m,dict):
            for k,v in m.items():
                if isinstance(v,str) and v.strip(): return k
    return None

def list_collections(chroma_dir):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    client = chromadb.PersistentClient(path=chroma_dir)
    cols = client.list_collections()
    print(f'  경로: {chroma_dir}')
    if not cols: print('  (컬렉션 없음)')
    for c in cols:
        try:
            col=client.get_collection(c.name); cnt=col.count()
            sample=col.get(limit=20, include=['metadatas'])['metadatas'] or []
            key=_detect_agency_key(sample); keys=list(sample[0].keys()) if sample else []
            print(f'  - {c.name:34} count={cnt:>8,}  | 기관키={key}  | 메타키={keys[:8]}')
        except Exception as e:
            print(f'  - {c.name:34} (count/메타 실패: {e})')

for tar_name in ['chroma_db.tar.gz', 'kh_v3_chroma_FIXED.tar.gz']:
    tar=f'{DRIVE}/{tar_name}'; tmp=f'/content/_probe_{tar_name}'
    if os.path.exists(tar):
        print(f'\n▶ tar: {tar_name}')
        shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
        with tarfile.open(tar) as t: t.extractall(tmp, filter='data')
        src=next((root for root,_,fs in os.walk(tmp) if 'chroma.sqlite3' in fs), None)
        print('  sqlite:', src)
        if src: list_collections(src)
    else:
        print(f'\n(없음) {tar}')
print('\n※ count>0 인 이름 → [1] COLL / 감지 기관키 → AGENCY_KEY')

In [ ]:
# [1] 마운트 + chroma 로컬 준비 (두 컬렉션 동시 — 빈 컬렉션 재사용 방지)
#  ┌──────────────────────────────────────────────────────────────────┐
#  │ MAIN = A/B/D/E (chunks_all),  CTYPE = C (kh_v3).                    │
#  │ 파일명/컬렉션명/EXPECT_MIN 이 [0b] 진단과 맞는지 확인.               │
#  └──────────────────────────────────────────────────────────────────┘
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
import shutil, tarfile, gc, chromadb
DRIVE = '/content/drive/MyDrive/data/bidmate'

# ===== MAIN: A/B/D/E = chunks_all =================================
MAIN = dict(
    TAG='chunks_all', CHUNK='chunks/chunks_all.json',
    BM25='bm25/bm25_index_bidmate_chunks_all_A-2.pkl',
    TAR='chroma_db.tar.gz', SUBDIR='chroma_db',
    COLL='bidmate_chunks_all', EXPECT_MIN=8000,
    AGENCY_KEY='agency', SIG_TH=0.5,
)
# ===== CTYPE: C = kh_v3 ===========================================
CTYPE = dict(
    TAG='kh_v3', CHUNK='chunks/kh_v3.json',
    BM25='bm25/bm25_index_bidmate_kh_v3_KURE_PHI.pkl',
    TAR='kh_v3_chroma_FIXED.tar.gz', SUBDIR='chroma_db_kh_v3_clean',
    COLL='bidmate_kh_v3_KURE_PHI', EXPECT_MIN=35000,
    AGENCY_KEY='auto', SIG_TH=0.5,
)
# ==================================================================

_AGENCY_CANDIDATES = ['agency','organization_cleaned','organization','agency_name','institution']
def _clear():
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
def _count(path, name):
    _clear()
    try: return chromadb.PersistentClient(path=path).get_collection(name).count()
    except Exception: return -1

def _ensure(d):
    LOCAL = f'/content/bidmate_{d["TAG"]}'
    CHROMA_DIR = f'{LOCAL}/{d["SUBDIR"]}'
    os.makedirs(LOCAL, exist_ok=True)
    coll = d['COLL'].strip()
    n = _count(CHROMA_DIR, coll)
    if n >= d['EXPECT_MIN']:
        print(f'[{d["TAG"]}] chroma 재사용: {CHROMA_DIR} (count={n:,})')
    else:
        print(f'[{d["TAG"]}] 재적재 (현재 count={n}) — 폴더 비우고 tar 재해제')
        shutil.rmtree(CHROMA_DIR, ignore_errors=True)
        tar=f'{DRIVE}/{d["TAR"]}'
        assert os.path.exists(tar), f'tar 없음: {tar}'
        sz=os.path.getsize(tar)/1e6
        assert sz>1, f'❌ tar 너무 작음({sz:.2f}MB): {tar}'
        print(f'  해제 중: {d["TAR"]} ({sz:.0f}MB)')
        with tarfile.open(tar) as t:
            assert any('chroma.sqlite3' in nm for nm in t.getnames()), '❌ tar 안에 sqlite 없음'
            t.extractall(LOCAL, filter='data')
        src=next((root for root,_,fs in os.walk(LOCAL) if 'chroma.sqlite3' in fs), None)
        assert src, '해제 후 sqlite 못 찾음'
        if os.path.abspath(src)!=os.path.abspath(CHROMA_DIR):
            shutil.rmtree(CHROMA_DIR, ignore_errors=True); shutil.move(src, CHROMA_DIR)
        n2=_count(CHROMA_DIR, coll)
        assert n2>=d['EXPECT_MIN'], f'❌ 해제 후에도 {coll} count={n2} (< {d["EXPECT_MIN"]}) — [0b] 로 이름 재확인'
        print(f'  재검증 통과: {coll} count={n2:,}')

    _cl=chromadb.PersistentClient(path=CHROMA_DIR)
    names=[c.name for c in _cl.list_collections()]
    assert coll in names, f'컬렉션 {coll} 없음. 존재: {names}'
    col=_cl.get_collection(coll); EXPECT_N=col.count()
    assert EXPECT_N>=d['EXPECT_MIN'], f'❌ EXPECT_N={EXPECT_N} 비정상'

    ak=d['AGENCY_KEY']
    if ak=='auto':
        metas=col.get(limit=30, include=['metadatas'])['metadatas'] or []
        ak=next((k for k in _AGENCY_CANDIDATES if any(isinstance(m,dict) and m.get(k) for m in metas)), None)
        if ak is None:
            for m in metas:
                if isinstance(m,dict):
                    ak=next((k for k,v in m.items() if isinstance(v,str) and v.strip()), None)
                    if ak: break
        assert ak, '❌ 기관키 자동감지 실패 — [0b] 확인 후 AGENCY_KEY 직접 지정'
    print(f'[{d["TAG"]}] 컬렉션={coll} | count={EXPECT_N:,} | 기관키={ak!r} | 그외={names}')

    for rel in [d['CHUNK'], d['BM25'], 'eval/eval_retrieval_579.csv']:
        s=f'{DRIVE}/{rel}'; dst=f'{LOCAL}/{rel}'
        assert os.path.exists(s), f'원본 없음: {s}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst): shutil.copy(s, dst)

    d.update(LOCAL=LOCAL, CHROMA_DIR=CHROMA_DIR, EXPECT_N=EXPECT_N, AGENCY_KEY=ak, COLL=coll)
    return d

MAIN  = _ensure(MAIN)
CTYPE = _ensure(CTYPE)

FIXED='/content/bidmate'
os.makedirs(f'{FIXED}/eval', exist_ok=True)
shutil.copy(f'{MAIN["LOCAL"]}/eval/eval_retrieval_579.csv', f'{FIXED}/eval/eval_retrieval_579.csv')
OUT_TAG_DIR=f'{FIXED}/outputs/routed_abde_chunksall__c_khv3'
os.makedirs(OUT_TAG_DIR, exist_ok=True)
print('\neval 동기화 / 통합 출력폴더:', OUT_TAG_DIR)
_MAIN, _CTYPE, _OUT_TAG_DIR = MAIN, CTYPE, OUT_TAG_DIR

In [ ]:
# [2] config 주입 — 공통값 + MAIN/CTYPE 분리값
import sys, types, os
from pathlib import Path
CODE='/content/drive/MyDrive/data/bidmate/code'
if CODE not in sys.path: sys.path.insert(0, CODE)
os.environ['HF_HOME']='/content/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

cfg=types.ModuleType('config')
cfg.ENV='colab'
# PROJECT_ROOT/CHUNKS/CHROMA/BM25 기본값은 MAIN 기준. 컬렉션별 실제 경로는 MAIN/CTYPE dict 로 들고감.
cfg.PROJECT_ROOT=Path(_MAIN['LOCAL'])
cfg.DATASET_DIR=cfg.PROJECT_ROOT
cfg.CHUNKS_PATH=Path(_MAIN['LOCAL'])/_MAIN['CHUNK']
cfg.CHROMA_PATH=Path(_MAIN['CHROMA_DIR'])
cfg.BM25_PATH=Path(_MAIN['LOCAL'])/_MAIN['BM25']
cfg.EVAL_PATH=Path('/content/bidmate/eval')
cfg.RESULT_DIR=cfg.PROJECT_ROOT/'eval_results'
cfg.ADAPTER_PATH=Path('/content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter')
cfg.LOG_PATH=cfg.PROJECT_ROOT/'web_user_access.log'
cfg.BASE_MODEL_ID='microsoft/Phi-4-mini-instruct'
cfg.LLM_MODEL='microsoft/Phi-4-mini-instruct'
cfg.EMBED_MODEL_ID='nlpai-lab/KURE-v1'
cfg.RERANKER_ID='BAAI/bge-reranker-v2-m3'
cfg.MAX_TOKENS_REWRITE=300; cfg.MAX_TOKENS_GENERATE=800
cfg.COLLECTION_NAME=_MAIN['COLL']
cfg.EXPECT_N=_MAIN['EXPECT_N']
cfg.OUT_TAG_DIR=_OUT_TAG_DIR
cfg.AGENCY_KEY=_MAIN['AGENCY_KEY']
cfg.SIG_TH=_MAIN['SIG_TH']
cfg.DENSE_K=15; cfg.SPARSE_K=15; cfg.RRF_K=60; cfg.TOP_K=5
cfg.MMR_LAMBDA=0.6; cfg.MMR_TOP_N=20; cfg.RERANK_TOP_N=15; cfg.BATCH_SIZE=64
# 라우팅용 컬렉션별 메타
cfg.MAIN=_MAIN; cfg.CTYPE=_CTYPE
cfg.ROUTE={'A':'MAIN','B':'MAIN','D':'MAIN','E':'MAIN','C':'CTYPE'}
sys.modules['config']=cfg
assert cfg.EXPECT_N>0, '❌ EXPECT_N=0 — [1] 재실행'
print('config OK')
print(f'  MAIN  col={_MAIN["COLL"]}  count={_MAIN["EXPECT_N"]:,}  agency={_MAIN["AGENCY_KEY"]!r}  sig={_MAIN["SIG_TH"]}')
print(f'  CTYPE col={_CTYPE["COLL"]}  count={_CTYPE["EXPECT_N"]:,}  agency={_CTYPE["AGENCY_KEY"]!r}  sig={_CTYPE["SIG_TH"]}')
print(f'  ROUTE {cfg.ROUTE}')
print(f'  OUT   {cfg.OUT_TAG_DIR}')

In [ ]:
# [3] pre-check — GPU + 파일 존재 + 두 컬렉션 count>0 최우선 검증
import torch, pickle, json, os, gc, chromadb
from pathlib import Path
import config as C

print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '❌ GPU 없음 — A100 런타임으로 변경'
gpu=torch.cuda.get_device_name(0); vram=round(torch.cuda.get_device_properties(0).total_memory/1e9,1)
print(f'  GPU : {gpu} | VRAM: {vram} GB')
if 'A100' not in gpu: print(f'  ⚠️ A100 아님({gpu}) — BATCH/속도 가정 다를 수 있음')
DEVICE='cuda'

def _ck(name, d):
    paths={'CHUNKS':Path(d['LOCAL'])/d['CHUNK'],'CHROMA':Path(d['CHROMA_DIR']),
           'BM25':Path(d['LOCAL'])/d['BM25']}
    for k,p in paths.items():
        print(f'  {"OK" if Path(p).exists() else "MISSING":8}{name}.{k:7}{p}')
_ck('MAIN', C.MAIN); _ck('CTYPE', C.CTYPE)
for k,p in {'EVAL':C.EVAL_PATH/'eval_retrieval_579.csv','ADAPTER':C.ADAPTER_PATH}.items():
    print(f'  {"OK" if Path(p).exists() else "MISSING":8}{k:11}{p}')

def _counts(d):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    n_chroma=chromadb.PersistentClient(path=d['CHROMA_DIR']).get_collection(d['COLL']).count()
    with open(Path(d['LOCAL'])/d['CHUNK'], encoding='utf-8') as f: n_chunks=len(json.load(f))
    with open(Path(d['LOCAL'])/d['BM25'],'rb') as f: n_bm25=len(pickle.load(f)['chunk_ids'])
    return n_chunks, n_chroma, n_bm25

for name,d in [('MAIN',C.MAIN),('CTYPE',C.CTYPE)]:
    nc,nh,nb=_counts(d)
    print(f'\n[{name}] 청크JSON {nc:,} | chroma {nh:,} | bm25 {nb:,} (기준 {d["EXPECT_N"]:,})')
    assert nh>0, f'❌ {name} chroma 비어있음 — [1] 재실행'
    assert nh==d['EXPECT_N'], f'❌ {name} chroma 불일치 {nh:,}!={d["EXPECT_N"]:,}'
    if nb!=nh: print(f'  ⚠️ bm25({nb:,})!=chroma({nh:,}) — 하이브리드 정합 확인')
if Path(C.ADAPTER_PATH).exists(): print('\n어댑터:', os.listdir(C.ADAPTER_PATH)[:6])
print('\n✅ pre-check 통과 (두 컬렉션 모두 비어있지 않음)')

In [ ]:
# [4] 서빙 모듈 로드 + retriever 2개(main/c) 조립 + 인메모리 패치
#   Drive 의 retrieval.py 는 구버전(생성자에 agency_meta_key/sig_th 없음) → 패치로 처리.
#   - _build_chroma_where : 키매핑 제거, 인스턴스별 _agency_key 사용
#   - retrieve           : D타입 sigmoid 컷오프(인스턴스별 _sig_th) + chunk id 폴백
#   - _normalize_chunk   : kh_v3 는 chunk_id 없음 → child_id/parent_id 로 폴백 (컬렉션별 토글)
import importlib.util, sys, pickle, gc, os, types, math
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import config as C

CODE='/content/drive/MyDrive/data/bidmate/code'
def load_module(name):
    path=f'{CODE}/{name}.py'; assert os.path.exists(path), f'파일 없음: {path}'
    spec=importlib.util.spec_from_file_location(name, path)
    mod=importlib.util.module_from_spec(spec); sys.modules[name]=mod
    spec.loader.exec_module(mod); return mod

Rtv=load_module('retrieval')
Rtv.DEVICE=DEVICE
print('retrieval 로드 OK')

# kh_v3 전용 _normalize_chunk (원본 호출 후 chunk_id 비면 child_id/parent_id 로 채움)
_orig_normalize = Rtv._normalize_chunk
def _normalize_khv3(c):
    out = _orig_normalize(c)
    if not out['chunk_id']:
        out['chunk_id'] = c.get('child_id') or c.get('parent_id') or out['metadata'].get('child_id','') or ''
    return out

# 공유 모델 1회 로드
embed_model=SentenceTransformer(C.EMBED_MODEL_ID, device=DEVICE, cache_folder='/content/hf_cache/hub')
reranker   =CrossEncoder(C.RERANKER_ID, device=DEVICE)
print('embed/reranker 공유 로드 OK')

def _open_collection(chroma_dir, coll, expect_n):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    col=chromadb.PersistentClient(path=str(chroma_dir)).get_collection(coll)
    cnt=col.count(); assert cnt>0 and cnt==expect_n, f'❌ {coll} count={cnt:,} (기대 {expect_n:,})'
    return col

def _apply_patches(r, agency_key, sig_th):
    r._agency_key = agency_key
    r._sig_th     = sig_th

    # (1) 키매핑 제거 — 감지된 기관키 그대로
    def _bcw(self, meta_filter):
        if not meta_filter: return None
        conds=[]
        for key,val in meta_filter.items():
            if not val: continue
            k = self._agency_key if key=='agency' else key
            if isinstance(val,dict):   conds.append({k: val})
            elif isinstance(val,list): conds.append({k: {"$in":[str(v) for v in val]}})
            else:                      conds.append({k: {"$eq": str(val)}})
        if not conds: return None
        return conds[0] if len(conds)==1 else {"$and": conds}
    r._build_chroma_where = types.MethodType(_bcw, r)

    # (2) retrieve — D타입 sigmoid 컷오프(self._sig_th) + id 폴백
    def _retrieve_sigcut(self, query, meta_filter=None, verbose=False):
        if meta_filter is None:
            meta_filter = Rtv.parse_metadata_filter(query)
        where   = self._build_chroma_where(meta_filter)
        allowed = self._filter_bm25_ids(meta_filter)
        subq    = self._decompose_query(query)
        if len(subq) > 1:
            dense_ids, sparse_ids = self._multi_retrieve(subq, where, allowed, original_query=query)
        else:
            dense_ids  = self._dense_search(query, where)
            sparse_ids = self._sparse_search(query, allowed)
        ranked  = self._rrf_fusion(dense_ids, sparse_ids)
        boosted = self._soft_boost(ranked)
        boosted = self._mmr_rerank(boosted, query=query)
        boosted = self._rerank(boosted, query=query)
        if len(subq) > 1:
            per=max(2, 5//len(subq)); cnt={}; top5=[]
            for cid,sc in boosted:
                meta=self.chunk_meta_map.get(cid,{})
                ag=meta.get(self._agency_key, meta.get('agency',''))
                if cnt.get(ag,0) < per:
                    top5.append((cid,sc)); cnt[ag]=cnt.get(ag,0)+1
                if len(top5)>=5: break
        else:
            top5 = boosted[:5]
        _sig=lambda x: 1/(1+math.exp(-x))
        if top5 and _sig(top5[0][1]) < self._sig_th:
            top5 = []   # 근거 부족 → D타입 거절
        def _g(m, cid, default):
            if cid in m: return m[cid]
            alt = int(cid) if str(cid).isdigit() else str(cid)
            return m.get(alt, default)
        return {
            "context"    : self._build_context(top5),
            "top_chunks" : [{"rank": i+1, "chunk_id": cid, "boosted_score": sc,
                             "text": _g(self.chunk_text_map, cid, ""),
                             "metadata": _g(self.chunk_meta_map, cid, {})}
                            for i,(cid,sc) in enumerate(top5)],
            "meta_filter": meta_filter, "dense_ids": dense_ids,
            "sparse_ids" : sparse_ids, "sub_queries": subq,
        }
    r.retrieve = types.MethodType(_retrieve_sigcut, r)
    return r

def build_retriever(d, use_khv3_norm):
    # 컬렉션별 _normalize_chunk 토글 후 load_chunks (load_chunks 는 모듈 전역 CHUNKS_PATH 참조)
    Rtv._normalize_chunk = _normalize_khv3 if use_khv3_norm else _orig_normalize
    chunks_path = os.path.join(d['LOCAL'], d['CHUNK'])
    _bak = Rtv.CHUNKS_PATH; Rtv.CHUNKS_PATH = chunks_path
    all_chunks = Rtv.load_chunks()
    Rtv.CHUNKS_PATH = _bak
    Rtv._normalize_chunk = _orig_normalize   # 원복(다음 빌드 오염 방지)

    col=_open_collection(d['CHROMA_DIR'], d['COLL'], d['EXPECT_N'])
    with open(os.path.join(d['LOCAL'], d['BM25']),'rb') as f: bm=pickle.load(f)
    r=Rtv.BidMateRetriever(
        collection=col, bm25_index=bm['index'], bm25_chunk_ids=bm['chunk_ids'],
        bm25_texts=bm['texts'], embed_model=embed_model, all_chunks=all_chunks, reranker=reranker,
    )
    _apply_patches(r, d['AGENCY_KEY'], d['SIG_TH'])
    r._tag=d['TAG']; r._collection=col
    # id 정합 점검: chroma 가 반환하는 id 가 map 키에 실제로 있는지
    _sample_ids = set(col.get(limit=5)['ids'])
    _hit = _sample_ids & set(r.chunk_text_map.keys())
    print(f'  [{d["TAG"]}] retriever 조립 OK (chunks={len(all_chunks):,}, agency_key={d["AGENCY_KEY"]!r}, '
          f'sig_th={d["SIG_TH"]}) | chroma↔map id 교집합 {len(_hit)}/5'
          + ('' if _hit else '  ⚠️ id 불일치 — text/metadata 빌 수 있음'))
    return r

retriever_main = build_retriever(C.MAIN,  use_khv3_norm=False)
retriever_c    = build_retriever(C.CTYPE, use_khv3_norm=True)

# ALL_AGENCIES = 두 컬렉션 기관명 합집합
_ag=set()
for r in (retriever_main, retriever_c):
    for m in r.chunk_meta_map.values():
        v=m.get(r._agency_key,'') or m.get('agency','')
        if v: _ag.add(v)
Rtv.ALL_AGENCIES=list(_ag)
print(f'✅ retriever 2개 준비 | ALL_AGENCIES={len(Rtv.ALL_AGENCIES)}')

# ── 라우팅 함수 ──────────────────────────────────────────────────
def set_active_retriever(r):
    Rtv.retriever = r          # get_context 가 참조하는 전역 교체

def route_retriever(qtype):
    return retriever_c if str(qtype).strip().upper()=='C' else retriever_main

set_active_retriever(retriever_main)   # 초기 전역 (generator init 시 참조 대비)
print('✅ 라우팅 준비 (C→kh_v3 / 그외→chunks_all)')

In [ ]:
# [5] generator 로드 (서빙 generation.py + FT Phi LoRA 어댑터)
#   generator.generate() 는 내부에서 Rtv.get_context() → 전역 Rtv.retriever 를 호출.
#   따라서 [7] 에서 행마다 set_active_retriever() 로 전역을 먼저 맞춰야 함.
import torch
Gen = load_module('generation')
generator = Gen.init_generator(Rtv.get_context)
Gen.generator = generator
print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,1),
      '/', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
print('✅ generator 초기화 완료 (Phi-4-mini + LoRA)')

In [ ]:
# [6] 스모크 — A/B/D/E 1건 + C 1건이 각각 올바른 retriever 로 검색·생성되는지
import time, pandas as pd, json, ast
eval_df=pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('타입 분포:', eval_df['type'].value_counts().to_dict())

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x] if isinstance(h,list) else []
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

def smoke(row, label):
    hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    r=route_retriever(row['type']); set_active_retriever(r)   # ★ 전역 스왑 먼저
    rewritten=generator._rewrite_query(row['question'], hist or None)
    rr=r.retrieve(rewritten, meta_filter=mf); top=rr.get('top_chunks',[])
    print(f'\n[{label} | type={row["type"]} | retriever={r._tag}] rewritten={rewritten[:45]!r}')
    print(f'  top_chunks: {len(top)}')
    if str(row['type']).upper()!='D':
        assert len(top)>0, f'❌ {label} 검색 0건 — chroma/메타필터/기관키 재확인'
    names=[c["metadata"].get("source_file","") for c in top]
    print('  retrieved_names:', json.dumps(names, ensure_ascii=False))
    if mf:
        where=r._build_chroma_where(mf)
        got=r._collection.get(where=where, limit=3)
        print(f'  [필터] where={where} → 매칭 {len(got["ids"])}건',
              '' if got['ids'] else '  ⚠️ 0건이면 mf 키/값이 chroma 메타와 불일치')
    t=time.time(); out=generator.generate(row['question'], history=hist or None, meta_filter=mf); dt=time.time()-t
    print('  답변:', out['answer'][:160]); print(f'  1건 {dt:.1f}초')
    return dt

row_main=eval_df[eval_df['type']!='C'].iloc[0]
dt1=smoke(row_main, 'MAIN')
csub=eval_df[eval_df['type']=='C']
dt2=smoke(csub.iloc[0], 'CTYPE') if len(csub) else None
print(f'\n579행 예상 ≈ {dt1*579/3600:.1f}시간 내외')

In [ ]:
# [7] 579행 생성 — 타입 라우팅(행마다 set_active_retriever) + question 기준 done 판정
import pandas as pd, json, ast, time, os
from tqdm.auto import tqdm
import config as _C

OUT=_C.OUT_TAG_DIR; os.makedirs(OUT,exist_ok=True)
GEN_PATH=f'{OUT}/e2e_kure_phi_ft_579.csv'

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('평가셋:', len(eval_df), '| 고유 question:', eval_df['question'].nunique(),
      '| 타입:', eval_df['type'].value_counts().to_dict())

done, records = set(), []
if os.path.exists(GEN_PATH):
    prev = pd.read_csv(GEN_PATH)
    prev = prev[prev['answer'].notna() & (prev['answer'].astype(str).str.len()>0)].drop_duplicates(subset='question')
    records = prev.to_dict('records'); done = set(prev['question'])
    print('체크포인트 재사용:', len(done))

pending = eval_df.drop_duplicates(subset='question')
pending = pending[~pending['question'].isin(done)]
print('신규:', len(pending))

for _, row in tqdm(pending.iterrows(), total=len(pending), desc='생성'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    active = route_retriever(row['type']); set_active_retriever(active)   # ★ 타입별 전역 스왑
    t0=time.time(); rewritten = generator._rewrite_query(q, hist or None)
    rr = active.retrieve(rewritten, meta_filter=mf); retr_ms = round((time.time()-t0)*1000)
    top = rr['top_chunks']
    t1=time.time(); out = generator.generate(q, history=hist or None, meta_filter=mf); gen_ms = round((time.time()-t1)*1000)
    records.append({
        'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'router':active._tag,
        'question':q,'rewritten_query':rewritten,
        'ground_truth_answer':row['ground_truth_answer'],'ground_truth_docs':row['ground_truth_docs'],
        'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('source_file','') for c in top], ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms,
    })
    if len(records)%25==0:
        pd.DataFrame(records).to_csv(GEN_PATH,index=False,encoding='utf-8-sig')

gen_df=pd.DataFrame(records).drop_duplicates(subset='question')
gen_df.to_csv(GEN_PATH,index=False,encoding='utf-8-sig')
print('✅ 생성 완료:', len(gen_df), '| 고유 question:', gen_df['question'].nunique())
print('라우팅 분포:', gen_df['router'].value_counts().to_dict())

In [ ]:
# [8] 생성 결과 무결성 점검 (question 기준 + 라우팅 정합)
import pandas as pd, json
import config as _C
GEN_PATH=f'{_C.OUT_TAG_DIR}/e2e_kure_phi_ft_579.csv'
df=pd.read_csv(GEN_PATH); ev=pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

print('저장된 행:', len(df), '| 고유 question:', df['question'].nunique())
empty_ans=df['answer'].isna().sum()+(df['answer'].astype(str).str.len()==0).sum()
print('answer 빈 행:', empty_ans)
print('eval 에 있는데 저장 안 된 question:', len(set(ev['question'])-set(df['question'])))
print('중복 question:', df['question'].duplicated().sum())

# 라우팅 정합: C는 kh_v3, 그외는 chunks_all
if 'router' in df.columns:
    bad=df[((df['type'].str.upper()=='C') & (df['router']!='kh_v3')) |
           ((df['type'].str.upper()!='C') & (df['router']!='chunks_all'))]
    print('라우팅 오배정 행:', len(bad))
    assert len(bad)==0, '❌ 라우팅 오배정 — route_retriever/[7] 확인'

# 생성 오류 행
err=df['answer'].astype(str).str.contains('오류', na=False).sum()
print('답변에 "오류" 포함 행:', err)

def _empty_names(raw):
    try: v=json.loads(raw)
    except Exception: return True
    return (not isinstance(v,list)) or len(v)==0 or all(not str(x).strip() for x in v)
n_empty=df['retrieved_names'].apply(_empty_names).sum()
print(f'retrieved_names 비어있는 행: {n_empty} / {len(df)} (D타입 거절 포함될 수 있음)')
assert n_empty < len(df)*0.5, '❌ 검색결과 절반 이상 비어있음 — chroma/메타필터 재점검'
print('✅ 무결성 통과')

In [ ]:
# [9] Retrieval 지표 — Hit@5 / MRR / nDCG (ground_truth_docs vs retrieved_names)
import pandas as pd, json, ast, math, os
import config as _C
OUT=_C.OUT_TAG_DIR
gen_df = pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv')

def _tolist(raw):
    if isinstance(raw,list): return raw
    for fn in (json.loads, ast.literal_eval):
        try:
            v=fn(raw)
            if isinstance(v,list): return v
        except Exception: pass
    return []
def _norm(x): return os.path.splitext(str(x).strip())[0].replace(' ','').lower()

def rmetrics(row, k=5):
    gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
    got=[_norm(x) for x in _tolist(row['retrieved_names'])][:k]
    if not gts: return None
    rank=next((i for i,g in enumerate(got,1) if g in gts), 0)
    hit=1.0 if rank else 0.0; mrr=1.0/rank if rank else 0.0
    dcg=sum(1.0/math.log2(i+1) for i,g in enumerate(got,1) if g in gts)
    idcg=sum(1.0/math.log2(i+1) for i in range(1,min(len(gts),k)+1))
    return pd.Series({'hit@5':hit,'mrr':mrr,'ndcg':dcg/idcg if idcg else 0.0})

rm = gen_df.join(gen_df.apply(rmetrics, axis=1))
valid = rm.dropna(subset=['hit@5'])
print(f'대상 {len(valid)}행 (정답문서 있는 행)')
print('전체:', valid[['hit@5','mrr','ndcg']].mean().round(4).to_dict())
print('\n타입별:\n', valid.groupby('type')[['hit@5','mrr','ndcg']].mean().round(4))

if valid['hit@5'].mean() == 0.0:
    print('\n⚠️ Hit@5 전체 0 — 매칭 진단 샘플:')
    for _, row in valid.head(3).iterrows():
        gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
        got=[_norm(x) for x in _tolist(row['retrieved_names'])][:5]
        print('  GT :', gts); print('  GOT:', got); print('  교집합:', set(gts)&set(got), '\n')

s = valid.groupby('type')[['hit@5','mrr','ndcg']].mean()
s.loc['ALL'] = valid[['hit@5','mrr','ndcg']].mean()
s.to_csv(f'{OUT}/retrieval_metrics_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [10] Generation Judge (gpt-5.4-mini async, 6지표, 50행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
from openai import AsyncOpenAI
from tqdm.auto import tqdm
import config as _C
nest_asyncio.apply()
OUT=_C.OUT_TAG_DIR

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'),'OPENAI_API_KEY 필요'
_M='gpt-5.4-mini'; _client=AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM=asyncio.Semaphore(15); _RETRY=3

_JP={
'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n"
                "[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n"
                "5점: 모든 내용이 Context 근거. 1점: Context 무관/날조.\n"
                "[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n"
             "[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n"
             "5점: 핵심을 정확·간결히 해결. 1점: 동문서답.\n"
             "[Question]\n{query}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n"
             "[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n"
             "5점: 근거 없으면 적절히 거절. 1점: 근거 없이 날조.\n"
             "[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n"
               "[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n"
               "5점: 모두 일치. 1점: 핵심 불일치.\n"
               "[Ground Truth]\n{ground_truth}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n"
                     "[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n"
                     "5점: 모두 필요. 1점: 대부분 불필요.\n"
                     "[Question]\n{query}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n"
                  "[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n"
                  "5점: 모든 핵심 포함. 1점: 누락 심각.\n"
                  "[Ground Truth]\n{ground_truth}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
}

def _parse(raw):
    if not raw: return None
    m=re.search(r'점수\s*:\s*(\d)',raw)
    if m: return int(m.group(1))
    s=raw.strip()
    if s.isdigit() and 1<=int(s)<=5: return int(s)
    d=re.findall(r'\b[1-5]\b',raw); return int(d[0]) if d else None

async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r=await _client.chat.completions.create(model=_M,
                    messages=[{'role':'user','content':prompt}],
                    max_completion_tokens=20, timeout=15)
                return _parse(r.choices[0].message.content)
            except Exception:
                if a==_RETRY-1: return None
                await asyncio.sleep(2**a)

async def score_one(q,ctx,ans,gt=None):
    tasks,none_keys={},[]
    for m in ('faithfulness','relevance','rejection'):
        tasks[m]=_ask(_JP[m].format(context=ctx,query=q,answer=ans))
    for m in ('correctness','context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m]=_ask(_JP[m].format(ground_truth=gt,answer=ans,context=ctx))
        else: none_keys.append(m)
    tasks['context_precision']=_ask(_JP['context_precision'].format(query=q,context=ctx))
    vals=await asyncio.gather(*tasks.values())
    res=dict(zip(tasks.keys(),vals))
    for k in none_keys: res[k]=None
    return res

_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
JUDGE_PATH=f'{OUT}/quant_scores_kure_phi_ft.csv'

async def run_judge():
    gdf=pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv').drop_duplicates(subset='question')
    done,rows=set(),[]
    if os.path.exists(JUDGE_PATH):
        ck=pd.read_csv(JUDGE_PATH); ck=ck[ck['relevance'].notna()].drop_duplicates(subset='question')
        rows=ck.to_dict('records'); done=set(ck['question']); print('judge 체크포인트:',len(done))
    pending=gdf[~gdf['question'].isin(done)]; print('judge 신규:',len(pending))
    for _,row in tqdm(pending.iterrows(), total=len(pending), desc='judge'):
        ans=row['answer']
        base={'id':row['id'],'question':row['question'],'type':row['type'],'difficulty':row['difficulty']}
        if not isinstance(ans,str) or '오류' in str(ans)[:30]:
            for m in _MET: base[m]=None
        else:
            base.update(await score_one(row['question'],row['retrieved_context'],ans,row.get('ground_truth_answer')))
        rows.append(base)
        if len(rows)%50==0: pd.DataFrame(rows).to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out=pd.DataFrame(rows).drop_duplicates(subset='question')
    out.to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    print('✅ judge 완료:',len(out)); return out

judge_df = asyncio.get_event_loop().run_until_complete(run_judge())

In [ ]:
# [11] Generation 요약
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
judge_df=pd.read_csv(f'{OUT}/quant_scores_kure_phi_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s=judge_df.groupby('type')[_MET].mean()
s.loc['ALL']=judge_df[_MET].mean()
s.round(3).to_csv(f'{OUT}/generation_summary_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [12] Release Gate
import pandas as pd
import config as _C
OUT=_C.OUT_TAG_DIR
retr=pd.read_csv(f'{OUT}/retrieval_metrics_kure_phi_ft.csv',index_col=0)
genm=pd.read_csv(f'{OUT}/generation_summary_kure_phi_ft.csv',index_col=0)
def v(x,p,g): return 'GOOD' if x>=g else ('PASS' if x>=p else 'FAIL')

print('RETRIEVAL (전체)')
print(f"  Hit@5 {retr.loc['ALL','hit@5']:.3f} → {v(retr.loc['ALL','hit@5'],0.90,0.95)}")
print(f"  MRR   {retr.loc['ALL','mrr']:.3f} → {v(retr.loc['ALL','mrr'],0.82,0.87)}")
print(f"  nDCG  {retr.loc['ALL','ndcg']:.3f} → {v(retr.loc['ALL','ndcg'],0.78,0.83)}")
print('타입별 MRR')
for t,(p,g) in {'A':(0.92,0.95),'B':(0.77,0.82),'C':(0.88,0.93),'D':(0.81,0.86),'E':(0.82,0.87)}.items():
    if t in retr.index: print(f"  {t} {retr.loc[t,'mrr']:.3f} → {v(retr.loc[t,'mrr'],p,g)}")
print('GENERATION (≥3.5 PASS / ≥4.0 GOOD)')
for m in ['faithfulness','relevance','rejection','context_precision']:
    if m in genm.columns:
        print(f"  {m:18} {genm.loc['ALL',m]:.3f} → {v(genm.loc['ALL',m],3.5,4.0)}")

In [ ]:
# [13] 산출물 목록
import os
import config as _C
OUT=_C.OUT_TAG_DIR
for f in sorted(os.listdir(OUT)):
    p=os.path.join(OUT,f)
    if os.path.isfile(p): print(f'{os.path.getsize(p)/1024:8.1f} KB  {f}')
    else:                 print(f'{"<dir>":>11}  {f}')

In [ ]:
# [14] 정성 분석 — 오류 역추적 / C타입 맥락 / 타입별 요약
import pandas as pd, os
import config as _C
OUT=_C.OUT_TAG_DIR
QUAL_DIR=f'{OUT}/qual'; os.makedirs(QUAL_DIR, exist_ok=True)

gen_df   = pd.read_csv(f'{OUT}/e2e_kure_phi_ft_579.csv').drop_duplicates(subset='question')
judge_df = pd.read_csv(f'{OUT}/quant_scores_kure_phi_ft.csv').drop_duplicates(subset='question')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
ERROR_TH=3.0

score_cols=['question']+[m for m in _MET if m in judge_df.columns]
merged=gen_df.merge(judge_df[score_cols], on='question', how='left')
print(f'병합: {len(merged)}행')

mask=pd.Series(False, index=merged.index)
for m in ['faithfulness','relevance']:
    if m in merged.columns: mask |= (merged[m].notna() & (merged[m]<=ERROR_TH))
mask |= merged['answer'].astype(str).str.contains('오류', na=False)
err_cols=[c for c in ['id','type','difficulty','router','question','ground_truth_answer',
                      'answer','retrieved_context']+_MET if c in merged.columns]
merged[mask][err_cols].to_csv(f'{QUAL_DIR}/qual_error_analysis.csv', index=False, encoding='utf-8-sig')
print(f'1) 오류 케이스: {int(mask.sum())}건 → qual_error_analysis.csv')

kws=['그 ','저 ','위에서','앞서','아까','해당','그것','거기']
cmask=(merged['type'].str.upper()=='C')
cmask |= merged['question'].astype(str).str.contains('|'.join(kws), na=False, regex=True)
c_cols=[c for c in ['id','type','router','question','ground_truth_answer','answer','retrieved_context'] if c in merged.columns]
merged[cmask][c_cols].to_csv(f'{QUAL_DIR}/qual_ctype_tracking.csv', index=False, encoding='utf-8-sig')
print(f'2) C타입/맥락 추적: {int(cmask.sum())}건 → qual_ctype_tracking.csv')

rows=[]
for t in ['A','B','C','D','E']:
    sub=merged[merged['type'].str.upper()==t]
    if sub.empty: continue
    r={'type':t,'n':len(sub)}
    for m in _MET: r[m]=round(sub[m].dropna().mean(),3) if m in sub.columns else None
    r['gen_errors']=int(sub['answer'].astype(str).str.contains('오류', na=False).sum())
    rows.append(r)
summary_df=pd.DataFrame(rows)
summary_df.to_csv(f'{QUAL_DIR}/qual_summary.csv', index=False, encoding='utf-8-sig')
print('3) 타입별 요약 → qual_summary.csv')
print('\n📊 타입별 요약'); print(summary_df.to_string(index=False))
print(f'\n✅ 정성 분석 저장: {QUAL_DIR}/')

In [ ]:
# [15] 산출물 Drive 백업 — outputs 통째로 (로컬은 런타임 종료 시 소실)
import shutil, os
import config as _C
if not os.path.ismount('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
SRC=_C.OUT_TAG_DIR
DST='/content/drive/MyDrive/data/bidmate/outputs/'+os.path.basename(SRC)
os.makedirs(os.path.dirname(DST), exist_ok=True)
if os.path.abspath(SRC)!=os.path.abspath(DST):
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
print('백업 완료:', DST)
for f in sorted(os.listdir(DST)): print('  -', f)